In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import product
from functools import partial
import importlib

import src.data
import src.features
import src.kernels
import src.krr
import src.svm
import src.metrics
import src.utils

from src.data import encode_labels, load_data, train_val_split
from src.features import DenseSIFT
from src.kernels import *
from src.krr import KernelRidgeRegression
from src.svm import KernelSVM
from src.metrics import accuracy
from src.utils import show_vector_image

In [2]:
DATA_DIR = "data/"

X_tr, X_te, y_tr_raw = load_data(DATA_DIR)

y_tr, inv_map = encode_labels(y_tr_raw)

n_classes = len(np.unique(y_tr))

### SIFT features

In [ ]:
dsift = DenseSIFT(patch_size=16, stride=16, num_cells=2, num_angles=4).fit(X_tr)

X_tr_sift = dsift.transform(X_tr)
X_te_sift = dsift.transform(X_te)

X_train_sift, X_val_sift, y_train_sift, y_val_sift = train_val_split(
    X_tr_sift, y_tr, test_size=0.3, seed=20
)

#### Support Vector Machine

Linear kernel grid search

In [ ]:
lrs = [0.6, 0.5, 0.4, 0.3, 0.2]
lams = [1e-4, 1e-5, 1e-6, 1e-7]

best = {"acc": -1.0, "lr": None, "lam": None}

for lr, lam in product(lrs, lams):
    kernel_fn = linear_kernel_matrix
    model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=500,
        lam=lam,
        patience=5,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lr={lr:.1e}, lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lr": lr, "lam": lam}

print("best:", best)

lr=6.0e-01, lam=1.0e-04, val_acc=0.2713
lr=6.0e-01, lam=1.0e-05, val_acc=0.3147
lr=6.0e-01, lam=1.0e-06, val_acc=0.3347
lr=6.0e-01, lam=1.0e-07, val_acc=0.3320
lr=5.0e-01, lam=1.0e-04, val_acc=0.2680
lr=5.0e-01, lam=1.0e-05, val_acc=0.3140
lr=5.0e-01, lam=1.0e-06, val_acc=0.3347
lr=5.0e-01, lam=1.0e-07, val_acc=0.3333
lr=4.0e-01, lam=1.0e-04, val_acc=0.2713
lr=4.0e-01, lam=1.0e-05, val_acc=0.3147
lr=4.0e-01, lam=1.0e-06, val_acc=0.3347
lr=4.0e-01, lam=1.0e-07, val_acc=0.3333
lr=3.0e-01, lam=1.0e-04, val_acc=0.2707
lr=3.0e-01, lam=1.0e-05, val_acc=0.3147
lr=3.0e-01, lam=1.0e-06, val_acc=0.3347
lr=3.0e-01, lam=1.0e-07, val_acc=0.3327
lr=2.0e-01, lam=1.0e-04, val_acc=0.2747
lr=2.0e-01, lam=1.0e-05, val_acc=0.3140
lr=2.0e-01, lam=1.0e-06, val_acc=0.3313
lr=2.0e-01, lam=1.0e-07, val_acc=0.3313
best: {'acc': 0.33466666666666667, 'lr': 0.6, 'lam': 1e-06}


Gaussian grid search

In [ ]:
lrs = [0.6, 0.5, 0.4, 0.3, 0.2]
lams = [1e-4, 1e-5, 1e-6, 1e-7]

gamma0 = estimate_gamma(X_train_sift)

best = {"acc": -1.0, "lr": None, "lam": None}

for lr, lam in product(lrs, lams):
    kernel_fn = partial(
        gaussian_kernel_matrix,
        gamma=gamma0,
    )
    model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=500,
        lam=lam,
        patience=5,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lr={lr:.1e}, lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lr": lr, "lam": lam}

print("best:", best)

lr=6.0e-01, lam=1.0e-04, val_acc=0.3587
lr=6.0e-01, lam=1.0e-05, val_acc=0.3760
lr=6.0e-01, lam=1.0e-06, val_acc=0.3713
lr=6.0e-01, lam=1.0e-07, val_acc=0.3713
lr=5.0e-01, lam=1.0e-04, val_acc=0.3587
lr=5.0e-01, lam=1.0e-05, val_acc=0.3753
lr=5.0e-01, lam=1.0e-06, val_acc=0.3747
lr=5.0e-01, lam=1.0e-07, val_acc=0.3747
lr=4.0e-01, lam=1.0e-04, val_acc=0.3580
lr=4.0e-01, lam=1.0e-05, val_acc=0.3733
lr=4.0e-01, lam=1.0e-06, val_acc=0.3733
lr=4.0e-01, lam=1.0e-07, val_acc=0.3733
lr=3.0e-01, lam=1.0e-04, val_acc=0.3580
lr=3.0e-01, lam=1.0e-05, val_acc=0.3727
lr=3.0e-01, lam=1.0e-06, val_acc=0.3727
lr=3.0e-01, lam=1.0e-07, val_acc=0.3727
lr=2.0e-01, lam=1.0e-04, val_acc=0.3580
lr=2.0e-01, lam=1.0e-05, val_acc=0.3713
lr=2.0e-01, lam=1.0e-06, val_acc=0.3713
lr=2.0e-01, lam=1.0e-07, val_acc=0.3713
best: {'acc': 0.376, 'lr': 0.6, 'lam': 1e-05}


Laplacian grid search

In [ ]:
lrs = [0.6, 0.5, 0.4, 0.3, 0.2]
lams = [1e-4, 1e-5, 1e-6, 1e-7]

gamma0 = estimate_laplacian_gamma(X_train_sift)

best = {"acc": -1.0, "lr": None, "lam": None}

for lr, lam in product(lrs, lams):
    kernel_fn = partial(
        laplacian_kernel_matrix,
        gamma=gamma0,
    )
    model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=500,
        lam=lam,
        patience=5,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lr={lr:.1e}, lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lr": lr, "lam": lam}

print("best:", best)

lr=6.0e-01, lam=1.0e-04, val_acc=0.3633
lr=6.0e-01, lam=1.0e-05, val_acc=0.3953
lr=6.0e-01, lam=1.0e-06, val_acc=0.3993
lr=6.0e-01, lam=1.0e-07, val_acc=0.3993
lr=5.0e-01, lam=1.0e-04, val_acc=0.3633
lr=5.0e-01, lam=1.0e-05, val_acc=0.3953
lr=5.0e-01, lam=1.0e-06, val_acc=0.4013
lr=5.0e-01, lam=1.0e-07, val_acc=0.4013
lr=4.0e-01, lam=1.0e-04, val_acc=0.3647
lr=4.0e-01, lam=1.0e-05, val_acc=0.3953
lr=4.0e-01, lam=1.0e-06, val_acc=0.3993
lr=4.0e-01, lam=1.0e-07, val_acc=0.3993
lr=3.0e-01, lam=1.0e-04, val_acc=0.3633
lr=3.0e-01, lam=1.0e-05, val_acc=0.3953
lr=3.0e-01, lam=1.0e-06, val_acc=0.4000
lr=3.0e-01, lam=1.0e-07, val_acc=0.4000
lr=2.0e-01, lam=1.0e-04, val_acc=0.3647
lr=2.0e-01, lam=1.0e-05, val_acc=0.3953
lr=2.0e-01, lam=1.0e-06, val_acc=0.3967
lr=2.0e-01, lam=1.0e-07, val_acc=0.3967
best: {'acc': 0.4013333333333333, 'lr': 0.5, 'lam': 1e-06}


Chi2 rbf grid search

In [ ]:
gamma0s = estimate_chi2_gammas_channel(X_train_sift)

lrs = [1.0, 0.8, 0.6]
lams = [5e-6, 1e-6, 5e-7]

best = {"acc": -1.0, "lr": None, "lam": None, "gamma": None}

for lr, lam in product(lrs, lams):
    kernel_fn = partial(
        chi2_rbf_kernel_matrix_channel,
        gammas=gamma0s,
        block_size=128,
        feature_block=64,
    )
    model = KernelSVM(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lr=lr,
        epochs=500,
        lam=lam,
        patience=5,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lr={lr:.1e}, lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lr": lr, "lam": lam}

print("best:", best)

lr=1.0e+00, lam=5.0e-06, val_acc=0.4107
lr=1.0e+00, lam=1.0e-06, val_acc=0.4127
lr=1.0e+00, lam=5.0e-07, val_acc=0.4127
lr=8.0e-01, lam=5.0e-06, val_acc=0.4107
lr=8.0e-01, lam=1.0e-06, val_acc=0.4153
lr=8.0e-01, lam=5.0e-07, val_acc=0.4153
lr=6.0e-01, lam=5.0e-06, val_acc=0.4107
lr=6.0e-01, lam=1.0e-06, val_acc=0.4153
lr=6.0e-01, lam=5.0e-07, val_acc=0.4153
best: {'acc': 0.41533333333333333, 'lr': 0.8, 'lam': 1e-06}


#### Kernel Ridge Regression

Linear kernel grid search

In [ ]:
lams = [1e-4, 5e-5, 1e-5, 5e-6, 1e-6]

best = {"acc": -1.0, "lam": None}

for lam in lams:
    kernel_fn = linear_kernel_matrix
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-04, val_acc=0.3207
lam=5.0e-05, val_acc=0.3260
lam=1.0e-05, val_acc=0.3320
lam=5.0e-06, val_acc=0.3320
lam=1.0e-06, val_acc=0.3187
best: {'acc': 0.332, 'lam': 1e-05}


Gaussian grid search

In [ ]:
lams = [1e-4, 1e-5, 1e-6, 1e-7]

gamma0 = estimate_gamma(X_train_sift)

best = {"acc": -1.0, "lam": None}

for lam in lams:
    kernel_fn = partial(
        gaussian_kernel_matrix,
        gamma=gamma0,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-04, val_acc=0.3833
lam=1.0e-05, val_acc=0.3860
lam=1.0e-06, val_acc=0.3480
lam=1.0e-07, val_acc=0.3313
best: {'acc': 0.386, 'lam': 1e-05}


Laplacian grid search

In [ ]:
lams = [1e-3, 5e-4, 1e-4, 5e-5]

gamma0 = estimate_laplacian_gamma(X_train_sift)

best = {"acc": -1.0, "lam": None}

for lam in lams:
    kernel_fn = partial(
        laplacian_kernel_matrix,
        gamma=gamma0,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-03, val_acc=0.3920
lam=5.0e-04, val_acc=0.3953
lam=1.0e-04, val_acc=0.4000
lam=5.0e-05, val_acc=0.3953
best: {'acc': 0.4, 'lam': 0.0001}


Chi2 rbf grid search

In [ ]:
gamma0s = estimate_chi2_gammas_channel(X_train_sift)

lams = [1e-3, 1e-4, 1e-5]

best = {"acc": -1.0, "lam": None, "gamma": None}

for lam in lams:
    kernel_fn = partial(
        chi2_rbf_kernel_matrix_channel,
        gammas=gamma0s,
        block_size=128,
        feature_block=64,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-03, val_acc=0.3900
lam=1.0e-04, val_acc=0.4147
lam=1.0e-05, val_acc=0.3933
best: {'acc': 0.4146666666666667, 'lam': 0.0001}


## After tuning parameters, train the best model on whole training set and use longer SIFT descriptors to maximise performance at test time.

In [ ]:
dsift = DenseSIFT(patch_size=16, stride=8, num_cells=2, num_angles=8).fit(X_tr)

X_tr_sift = dsift.transform(X_tr)
X_te_sift = dsift.transform(X_te)

X_train_sift, X_val_sift, y_train_sift, y_val_sift = train_val_split(
    X_tr_sift, y_tr, test_size=0.3, seed=20
)

Check validation error with longer SIFT descriptor

In [7]:
gamma0s = estimate_chi2_gammas_channel(X_train_sift)

lams = [1e-4]

best = {"acc": -1.0, "lam": None, "gamma": None}

for lam in lams:
    kernel_fn = partial(
        chi2_rbf_kernel_matrix_channel,
        gammas=gamma0s,
        block_size=128,
        feature_block=64,
    )
    model = KernelRidgeRegression(
        n_classes=n_classes,
        kernel_fn=kernel_fn,
        lam=lam,
    ).fit(X_train_sift, y_train_sift, X_val=X_val_sift, y_val=y_val_sift)

    pred_val, _ = model.predict(X_val_sift)
    acc = accuracy(y_val_sift, pred_val)
    print(f"lam={lam:.1e}, val_acc={acc:.4f}")

    if acc > best["acc"]:
        best = {"acc": acc, "lam": lam}

print("best:", best)

lam=1.0e-04, val_acc=0.5500
best: {'acc': 0.55, 'lam': 0.0001}


Actually train on whole training set with longer SIFT descriptor, and predict on test set.

In [ ]:
gamma0s = estimate_chi2_gammas_channel(X_train_sift)

lam = 1e-4

kernel_fn = partial(
    chi2_rbf_kernel_matrix_channel,
    gammas=gamma0s,
    block_size=128,
    feature_block=64,
)
best_model = KernelRidgeRegression(
    n_classes=n_classes,
    kernel_fn=kernel_fn,
    lam=lam,
).fit(X_tr_sift, y_tr)

In [ ]:
yte_int, _ = best_model.predict(X_te_sift)
yte = np.array([inv_map[i] for i in yte_int])

sub = pd.DataFrame({"Prediction": yte})
sub.index += 1
sub.to_csv("results/submission.csv", index_label="Id")

In [50]:
import sounddevice as sd

duration = 3
frequency = 200
sample_rate = 4400

t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
wave = 0.5 * np.sin(2 * np.pi * frequency * t)

sd.play(wave, sample_rate)